# Autorização SP/SADT

Monte e valide uma solicitação sintética, avalie regras locais e exercite as respostas offline.

In [1]:
from copy import deepcopy

from voa_tiss import (
    build_autorizacao,
    check_autorizacao,
    examples,
    parse_autorizacao,
    validate,
)

solicitacao = parse_autorizacao(examples.load("autorizacao-solicitacao"))
xml_solicitacao = build_autorizacao(solicitacao)
assert validate(xml_solicitacao).valid
len(xml_solicitacao)

2311

In [2]:
contexto = {"operadora": "000000"}
solicitacao_sem_regra = deepcopy(solicitacao)
solicitacao_sem_regra["solicitacao"]["procedimentos_solicitados"][0]["procedimento"][
    "codigo_procedimento"
] = "40901220"
solicitacao_pendente = deepcopy(solicitacao)
solicitacao_pendente["solicitacao"].pop("indicacao_clinica")
decisoes = [
    check_autorizacao(item, contexto=contexto).decisao["tipo"]
    for item in (solicitacao_sem_regra, solicitacao_pendente, solicitacao)
]
decisoes

['aprovar_por_regra', 'pendente', 'encaminhar_auditor']

In [3]:
from voa_tiss.sandbox import responder_autorizacao

respostas = {
    cenario: responder_autorizacao(solicitacao, cenario=cenario)
    for cenario in ("aprovado", "pendente", "negado")
}
assert all(validate(xml).valid for xml in respostas.values())
{
    cenario: parse_autorizacao(xml)["autorizacao"]["status_solicitacao"]
    for cenario, xml in respostas.items()
}

{'aprovado': '1', 'pendente': '5', 'negado': '3'}

In [4]:
from voa_tiss import parse_demonstrativo
from voa_tiss.sandbox import demonstrativo, enviar_lote

envio = enviar_lote(examples.load("lote-valido"), registro_ans="000000")
resultado = parse_demonstrativo(demonstrativo(envio["protocolo"]))
[(item.guia_ref, item.motivo) for item in resultado]

[('0001',
  '1818; 1818 - Procedimento exige autorização prévia. (regra exige-autorizacao-consulta)'),
 ('0002',
  '1818; 1818 - Procedimento exige autorização prévia. (regra exige-autorizacao-consulta)')]